### 📘 学习打卡

##### 请执行下方 cell，点击窗口中的“开始学习”按钮开始本次学习，完成学习后请点击“结束学习”按钮。开始和结束时间都会记录到统一日志文件中，帮助对你的学习情况进行分析。

In [1]:
from projects.clock_in import create_session_widgets
display(create_session_widgets("projects/sdt.log"))

# 语法制导翻译

---

## 概览

本作业将引导你完成语法制导翻译的学习与练习。你将从一个简单的表达式翻译问题出发，逐步掌握语法制导定义（SDD）和翻译模式（TC）的设计方法。

## 学习目标

1. 理解**语法制导翻译**的基本概念：属性、语义规则、综合属性
2. 掌握**语法制导定义（SDD）**的设计方法与执行过程
3. 理解**翻译模式（TC）**的设计思路与执行过程

---

## 1. 语法制导定义

### 1.1 基本概念

#### 语法制导翻译概述

在上一章中，我们学习了**上下文无关文法（CFG）**，它用于描述语言的语法结构——即哪些单词串是合法的。

然而，编译器不仅要判断程序的语法是否正确，还要**将源程序翻译为目标代码**。**语法制导翻译（Syntax-Directed Translation）** 正是实现这一目标的核心方法。

**基本思想**：
- 用文法描述源语言的**语法结构**（产生式）
- 为每个语法结构附加**语义信息**，描述如何翻译/执行它
- 通过遍历语法分析树完成翻译（语法分析之后或同时）

在本章中，我们以**中缀表达式 → 后缀表达式**的翻译为例。例如：

```
9 - 5 + 2   →   9 5 - 2 +
```

#### 语法制导定义（SDD）

**语法制导定义（Syntax-Directed Definition, SDD）** 是语法制导翻译的形式化描述框架，它由两部分组成：

1. **上下文无关文法（CFG）**：描述语法结构
2. **语义规则（Semantic Rules）**：描述如何计算出翻译结果

具体来说，SDD 为：
- **每个文法符号**（终结符或非终结符）关联一组**属性（Attribute）**，用于存储翻译相关的信息（如类型、代码串、内存位置等）
- **每个产生式**关联一组**语义规则**，用于计算属性值

**综合属性（Synthesized Attribute）**：

在 SDD 中，有一类重要的属性称为**综合属性**。综合属性的特点是：
- 节点的属性值由**其孩子节点**的属性值决定
- 可以通过**自底向上**的方式计算（先计算孩子节点，再计算父节点）

**执行过程**：

给定一个输入单词串，SDD 的翻译过程如下：

1. 根据文法进行**语法分析**，构建**语法分析树**
2. 在语法分析树的每个节点上，利用其对应的语义规则计算属性值
3. 最终得到**注释语法分析树（Annotated Parse Tree）**——即标注了所有属性值的语法树
4. 根节点的属性值即为整个输入的翻译结果

这个过程可以直观地理解为：输入单词串 → 语法树 → 标注属性值（注释语法树）→ 输出翻译结果。

#### 💡 思考题 1.1

**问题：**

> 1. 语法制导定义（SDD）由哪两部分组成？综合属性的值是如何计算的？
> 2. 为什么说综合属性的计算是"自底向上"的？请结合语法分析树的结构简要说明。

请运行下面的 cell，在 AI 助教窗口中回答这两个问题。

In [2]:
%run projects/problem_ai_helper.py 1.1

D:\code_warehouse\Curriculum\Compiler Systems\sdt\projects\problem_ai_helper.py:218: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  self.input_box.on_submit(self.send_message)


<IPython.core.display.Javascript object>

---

### 1.2 语法制导定义的设计

#### 设计思想

设计语法制导定义的核心思想与设计文法类似：**从"人会做"到"让计算机会做"**。

**第一步：人会做**

首先，我们自己要清楚如何完成翻译。以中缀表达式到后缀表达式的翻译为例：

```
9 - 5 + 2   →   9 5 - 2 +
```

人会怎么做？
- 如果是**变量或常量**：后缀形式就是它自身，即 `Postfix(E) = E`
- 如果是**二元运算** `E = E1 op E2`：先翻译 E1，再翻译 E2，最后输出 op，即 `Postfix(E) = Postfix(E1) Postfix(E2) op`
- 如果是**括号表达式** `E = (E1)`：后缀形式就是 E1 的后缀形式，即 `Postfix(E) = Postfix(E1)`

体会：我们不是在为某个特定单词串设计翻译方法，而是设计一种**通用的方法**，可对所有合法的单词串进行翻译。

为此，我们实际上是为每种语法结构设计翻译方法；对给定的任意单词串，分析其是哪些语法结构的组合；再对每个语法结构应用所设计的翻译方法，最终即得到完整的翻译结果。

例如，`9 - 5 + 2` 的翻译过程：

```
Postfix(9 - 5 + 2)
  = Postfix(9 - 5) Postfix(2) + （语法结构2的翻译方法）
  = Postfix(9) Postfix(5) - 2 + （语法结构2的翻译方法）
  = 9 5 - 2 +                   （语法结构1的翻译方法）
```

**第二步：让计算机会做——符号化**

将上述翻译方法形式化为 SDD：

1. **为文法符号设计属性**：每个非终结符需要一个属性来存储其后缀形式。例如，为 `expr` 设计属性 `t`（string 类型），`expr.t` 表示该表达式对应的后缀串。

2. **为每个产生式编写语义规则**：将翻译方法转化为属性之间的运算。

以表达式文法为例：

```
expr  → expr + term | expr - term | term
term  → 0 | 1 | 2 | ... | 9
```

对应的 SDD 如下：

| 产生式 | 语义规则 |
|--------|----------|
| `expr → expr1 + term` | `expr.t = expr1.t \|\| term.t \|\| '+'` |
| `expr → expr1 - term` | `expr.t = expr1.t \|\| term.t \|\| '-'` |
| `expr → term` | `expr.t = term.t` |
| `term → 0` | `term.t = '0'` |
| `term → 1` | `term.t = '1'` |
| `...` | `...` |
| `term → 9` | `term.t = '9'` |

其中 `||` 表示字符串连接操作。

> 💡 注意：`expr → expr1 + term` 中的 `expr1` 表示产生式右部的第一个 `expr`，用于与左部的 `expr` 区分，并非一个不同的非终结符。

**注释语法分析树**：

在语法分析树的每个节点上标注属性值后（通过后序遍历计算），就得到了**注释语法分析树（Annotated Parse Tree）**。

对于输入 `9-5+2`，其注释语法分析树如下图所示（属性值为 `t`，即后缀串）：

```
                    expr (t = "95-2+")
                   /    |    \
               expr     -    term
            (t="95-")        (t="2")
            /   |   \          |
         expr   -   term       2
      (t="9")     (t="5")
         |          |
        term        5
         |
         9
```

通过自底向上的计算，根节点的 `expr.t = "95-2+"` 即为最终翻译结果。

#### 💡 思考题 1.2

**问题：**

> 给定无二义表达式文法，请完成以下两个任务：

> **任务 1：补全语义规则**

> 文法如下：
> ```
> expr  → expr + term | expr - term | term
> term  → term * factor | term / factor | factor
> factor → digit | ( expr )
> digit → 0 | 1 | 2 | ... | 9
> ```

> 请为每个产生式设计语义规则——在下方第一个 cell 中每个产生式最后的花括号 `{ }` 内填入属性计算公式，实现表达式中缀形式转换为后缀形式。
> 
> 格式示例：`{ expr.t = expr1.t + term.t + '+' }`

> **任务 2：属性计算过程**

> 对表达式 `7 - 3 * 4`，按**后序遍历**顺序，逐步计算每个节点的属性值。

> 请在下方第二个 cell 中的 `attr_steps` 列表内（TODO 所示位置）按顺序输入每一步的计算。
> 
> 格式示例：`"digit.t = '7'"`

> 在下方每个 cell 填写完答案后，请执行 cell，你的答案会保存到文件中。然后可运行 AI 助教 cell，与助教讨论你的答案。如修改了语法制导定义、翻译过程，重新保存后，需重新执行 AI 助教 cell，以便基于新的解答进行讨论。

In [15]:
import datetime

semantic_rules = [
    # 请在每个产生式右部的花括号 {} 内填入语义规则
    # 格式：左部非终结符属性 = 右部符号属性计算
    # 例如：expr.t = expr1.t + term.t + '+'
    "expr -> expr1 + term { expr.t = expr1.t + term.t + '+' }",
    "expr -> expr1 - term { expr.t = expr1.t + term.t + '-' }",
    "expr -> term { expr.t = term.t }",
    "term -> term1 * factor { term.t = term1.t + factor.t + '*' }",
    "term -> term1 / factor { term.t = term1.t + factor.t + '/' }",
    "term -> factor { term.t = factor.t }",
    "factor -> digit { factor.t = digit.t }",
    "factor -> ( expr ) { factor.t = expr.t }",
    "digit -> 0 { digit.t = '0' }",
    "digit -> 1 { digit.t = '1' }",
    "digit -> 2 { digit.t = '2' }",
    "digit -> 3 { digit.t = '3' }",
    "digit -> 4 { digit.t = '4' }",
    "digit -> 5 { digit.t = '5' }",
    "digit -> 6 { digit.t = '6' }",
    "digit -> 7 { digit.t = '7' }",
    "digit -> 8 { digit.t = '8' }",
    "digit -> 9 { digit.t = '9' }",
]

# 将语义规则保存到文件，供 AI 助教读取
with open("projects/semantic_rules.txt", "w", encoding="utf-8") as f:
    for rule in semantic_rules:
        f.write(rule + "\n")

timestamp = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
with open("projects/sdt.log", "a", encoding="utf-8") as f:
    f.write("\n#################### 事件分隔 ####################\n")
    f.write(f"=== [{timestamp}] 思考题1.2 - 语义规则 ===\n")
    f.write("\n".join(semantic_rules))
    f.write("\n\n")

print("✅ 语义规则已保存到 projects/semantic_rules.txt，日志已追加到 sdt.log")


✅ 语义规则已保存到 projects/semantic_rules.txt，日志已追加到 sdt.log


In [18]:
import datetime
"""
7 - 3 * 4
7 - 3 4 *
7 3 - 4 *
"""
attr_steps = [
    # 请按后序遍历顺序，每行输入一个属性的计算
    # 格式：node.t = 公式 = 'value'
    # 例如：digit.t = '7'
    "digit.t = '7'",
    "factor.t = '7'",
    "term.t = '7'",
    "digit.t = '3'",
    "digit.t = '4'",
    "term.t = '3 4 *'",
    "expr.t = '7 3 4 * -'",
    # TODO: 继续输入后续步骤...
]

# 将属性计算步骤保存到文件，供 AI 助教读取
with open("projects/attr_steps.txt", "w", encoding="utf-8") as f:
    for step in attr_steps:
        f.write(step + "\n")

timestamp = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
with open("projects/sdt.log", "a", encoding="utf-8") as f:
    f.write("\n#################### 事件分隔 ####################\n")
    f.write(f"=== [{timestamp}] 思考题1.2 - 属性计算步骤 ===\n")
    f.write("\n".join(attr_steps))
    f.write("\n\n")

print("✅ 属性计算步骤已保存到 projects/attr_steps.txt，日志已追加到 sdt.log")


✅ 属性计算步骤已保存到 projects/attr_steps.txt，日志已追加到 sdt.log


#### 💡 需要 AI 帮助？

如果你已完成两个任务，运行下方的 AI 助教单元格。它会读取你的答案，分析语义规则和属性计算步骤是否正确，并给出引导性的反馈。

In [19]:
%run projects/code_ai_helper.py sdd_exercise

D:\code_warehouse\Curriculum\Compiler Systems\sdt\projects\code_ai_helper.py:361: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  self.input_box.on_submit(self.send_message)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 2. 翻译模式

### 2.1 从语法制导定义到翻译模式

在上一单元中，我们学习了**语法制导定义（SDD）**——通过属性 + 语义规则来描述"要计算什么"。这种方式是**声明式**的，它规定了属性值的计算方法，但没有规定计算的顺序。

然而，在真正的编译器中，我们需要明确指定"什么时候做什么"。**翻译模式（Translation Scheme, TC）** 正是为此而生。

**SDD vs TC**：

| | 语法制导定义（SDD） | 翻译模式（TC） |
|:---|:---|:---|
| 风格 | 声明式 | 过程式 |
| 核心 | 产生式附加语义规则（属性计算） | 语义动作嵌入产生式右部 |
| 计算顺序 | 未规定（由属性依赖关系决定） | 明确规定（语义动作的位置决定执行顺序） |
| 实现 | 遍历语法分析树计算属性 | 遍历语法分析树执行语义动作 |

**TC 的核心思想**：将语义动作（Semantic Action）嵌入到产生式的右部，用花括号 `{ }` 括起来，表示在语法分析过程中的某个时刻执行该动作。

例如，后缀表达式转换的 SDD 语义规则：

```
expr → expr1 + term   { expr.t = expr1.t || term.t || '+' }
```

对应的 TC 将语义动作嵌入右部：

```
expr → expr1 + term   { print('+') }
```

#### TC 的执行

如果已经设计好了一个 TC，它是如何执行的？

**执行方式**：对输入串进行语法分析，构建语法分析树，然后遍历语法树。遍历过程中遇到语义动作（花括号中的代码）即执行它。

> 语义动作可以看作语法树中的特殊节点。后序遍历时，遇到普通节点继续遍历其子树，遇到语义动作节点则执行它。

**设计 TC 需要回答的两个核心问题**：

1. **打印什么？** —— 每个语义动作应该输出什么内容？
2. **嵌入到什么位置？** —— 语义动作应该放在产生式右部的哪个位置？

直接回答这两个问题可能没有头绪。我们可以换一个思路：**从执行过程反推设计**。

### 2.2 翻译模式设计思路

假设我们已经有了一棵语法分析树，现在要思考：应该在哪里执行什么语义动作，才能得到正确的后缀表达式？

以表达式 `9 - 5 + 2` 的语法树为例：

```
                       expr
                   /    |    \
                 expr   +    term
              /   |   \         |
           expr   -   term      2
            |          |
          term         5
            |
            9
```

**目标**：在这棵语法树上添加语义动作，使得对该树的后序遍历能够输出 `95-2+`。

**思考**：

1. **运算数（叶节点）应该打印什么？** 答案：打印运算数自身。例如，叶节点 `9` 应打印 `'9'`。

2. **运算数应该何时打印？** 答案：在遍历到该叶节点时立即打印，打印动作可放在该叶节点的右侧。

3. **运算符应该打印什么？** 答案：只打印运算符自身。例如，`-` 节点应打印 `'-'`。注意，不必关心两个运算对象的打印，因为**在各自的子树遍历时已打印出来**。

4. **运算符应该何时打印？** 答案：对于后缀表达式，运算符需要放在两个运算对象之后。因此在语法树中，运算符的打印动作应放在其**右兄弟（第二个运算对象）的子树遍历完成之后**。

于是，插入了语义动作的语法分析树应该像下面这样：

```
                                expr
                      /    /          \      \
                   expr   +          term   {print('+')}
              /  /  |   \            /  \
           expr - term {print('-')} 2  {print('2')}
            |    /    \
          term  5 {print('5')}
        /      \
       9   {print('9')}
```

通过这个反推过程，我们自然得到了设计 TC 的两个核心问题的答案：

- **打印什么**：运算数打印自身，运算符打印自身。
- **嵌入位置**：运算数在自身右侧（语法树中成为右兄弟，产生式中成为下一个成员）；运算符在右兄弟的子树之后（语法树中成为父节点最后一个孩子，产生式中成为右部最后一个成员）。


### 2.3 动手练习：设计前缀表达式的 TC

前缀表达式的定义：
- 变量/常量：前缀形式就是它自身，即 `Prefix(E) = E`
- 二元运算 `E = E1 op E2`：`Prefix(E) = op Prefix(E1) Prefix(E2)`
- 括号表达式 `E = (E1)`：`Prefix(E) = Prefix(E1)`

**问题：**

> 给定无二义表达式文法，请完成以下两个任务：

> **任务 1：设计翻译模式（TC）**

> 文法如下：
> ```
> expr  → expr + term | expr - term | term
> term  → term * factor | term / factor | factor
> factor → digit | ( expr )
> digit → 0 | 1 | 2 | ... | 9
> ```

> 请为文法设计 TC（翻译模式），实现表达式中缀形式转换为后缀形式，在下方第一个 cell 中每个产生式右部正确的位置插入正确的语义动作，用花括号包围。
> 
> 格式示例：`{ print('+') }`

> **任务 2：执行翻译过程**

> 对表达式 `9 - 5 * 2`，按**后序遍历**顺序，逐步写出执行过程——即每个语义动作被触发时打印的内容，最终得到的完整前缀表达式。

> 请在下方第二个 cell 中的 `exec_steps` 列表内按顺序输入每一步的打印内容。
> 
> 格式示例：`"print '+'"`

> 在下方两个 cell 填写完答案后，请执行 cell，你的答案会保存到文件中。随后可与 AI 助教讨论，如答案有错可进行修改、再与 AI 助教讨论，直至得到正确结果。

In [23]:
import datetime

translation_schemes = [
    # 请在每个产生式右部恰当位置嵌入花括号 {} 包围的语义动作
    # 例如：{ print('+') }
    "expr -> {print('+')} expr1 + term",
    "expr -> {print('-')} expr1 - term",
    "expr -> term",
    "term -> {print('*')} term1 * factor",
    "term -> {print('/')} term1 / factor",
    "term -> factor",
    "factor -> digit",
    "factor -> ( expr )",
    "digit -> 0 {print('0')}",
    "digit -> 1 {print('1')}",
    "digit -> 2 {print('2')}",
    "digit -> 3 {print('3')}",
    "digit -> 4 {print('4')}",
    "digit -> 5 {print('5')}",
    "digit -> 6 {print('6')}",
    "digit -> 7 {print('7')}",
    "digit -> 8 {print('8')}",
    "digit -> 9 {print('9')}",
]

with open("projects/translation_schemes.txt", "w", encoding="utf-8") as f:
    for rule in translation_schemes:
        f.write(rule + "\n")

timestamp = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
with open("projects/sdt.log", "a", encoding="utf-8") as f:
    f.write("\n#################### 事件分隔 ####################\n")
    f.write(f"=== [{timestamp}] 思考题2.2 - 翻译模式 ===\n")
    f.write("\n".join(translation_schemes))
    f.write("\n\n")

print("✅ 翻译模式已保存到 projects/translation_schemes.txt，日志已追加到 sdt.log")


✅ 翻译模式已保存到 projects/translation_schemes.txt，日志已追加到 sdt.log


In [32]:
import datetime

exec_steps = [
    # 9-5*2
    # 请按后序遍历顺序，每行输入一个语义动作的执行，用双引号包围，末尾逗号分隔
    # 例如："print '-'",
    # TODO: 输入执行步骤...
    "print '-'",
    "print '9'",
    "print '*'",
    "print '5'",
    "print '2'",
]

with open("projects/exec_steps.txt", "w", encoding="utf-8") as f:
    for step in exec_steps:
        f.write(step + "\n")

timestamp = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
with open("projects/sdt.log", "a", encoding="utf-8") as f:
    f.write("\n#################### 事件分隔 ####################\n")
    f.write(f"=== [{timestamp}] 思考题2.2 - 执行过程 ===\n")
    f.write("\n".join(exec_steps))
    f.write("\n\n")

print("✅ 执行步骤已保存到 projects/exec_steps.txt，日志已追加到 sdt.log")


✅ 执行步骤已保存到 projects/exec_steps.txt，日志已追加到 sdt.log


#### 💡 需要 AI 帮助？

如果你已完成两个任务，运行下方的 AI 助教 cell。它会读取你的答案，分析翻译模式和执行过程是否正确，并给出引导性的反馈。如果修改了翻译模式和翻译过程，重新保存后，需重新执行这个 cell，以便基于新的解答进行讨论。

In [33]:
%run projects/code_ai_helper.py tc_exercise

D:\code_warehouse\Curriculum\Compiler Systems\sdt\projects\code_ai_helper.py:361: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  self.input_box.on_submit(self.send_message)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 3. 总结

本 Notebook 介绍了编译原理中语法制导翻译的核心内容：

1. **语法制导定义（SDD）**：属性 + 语义规则，综合属性自底向上计算
2. **翻译模式（TC）**：语义动作嵌入产生式右部，遍历语法树时执行
3. **设计方法论**：从“人会做”到“让计算机会做”——先理清翻译方法，再形式化、再实现